# 02 Target Definition

Este notebook define o target final para modelagem, divide as bases de treino e OOT, e salva os conjuntos de dados finais.


## 1. Imports e Configurações

Definimos o período de treino e OOT.

In [10]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path('..').resolve()))

from src.population import build_active_population
from src.target import build_contract_target, build_population_target, choose_target_definition

RANDOM_STATE = 42
TRAIN_PERIODS = [
    "2020-01", "2020-02", "2020-03", "2020-04", "2020-05", "2020-06", 
    "2020-07", "2020-08", "2020-09", "2020-10", "2020-11", "2020-12", 
    "2021-01", "2021-02", "2021-03", "2021-04", "2021-05", "2021-06", 
    "2021-07", "2021-08", "2021-09", "2021-10", "2021-11", "2021-12", 
    "2022-01", "2022-02", "2022-03", "2022-04", "2022-05", "2022-06", 
    "2022-07", "2022-08", "2022-09", "2022-10", "2022-11", "2022-12", 
    "2023-01", "2023-02", "2023-03", "2023-04", "2023-05", "2023-06",
    "2023-07", "2023-08", "2023-09", "2023-10", "2023-11", "2023-12",
]
OOT_PERIODS = [ 
    "2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06", 
    "2024-07", "2024-08", "2024-09", "2024-10", "2024-11", "2024-12",
    "2025-01",
]

PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("TRAIN_PERIODS desde", TRAIN_PERIODS[0], "até", TRAIN_PERIODS[-1])
print("OOT_PERIODS desde", OOT_PERIODS[0], "até", OOT_PERIODS[-1])


TRAIN_PERIODS desde 2020-01 até 2023-12
OOT_PERIODS desde 2024-01 até 2025-01


## 2. Carregamento das Bases

Carregamos apenas a população ativa, que representa o universo elegível para modelagem.

As informações necessárias para construção dos targets são carregadas internamente pelas funções do módulo `target.py`, evitando duplicidade de processamento no notebook.

In [11]:
population_active = build_active_population()

print("population_active", population_active.shape)


population_active (81882, 28)


## 3. Construção do Target

A definição do target é realizada em nível de cliente, utilizando o histórico de pagamento dos contratos presentes na população ativa.

O processo é composto pelas seguintes etapas:

* Inicialmente são calculados os atrasos observados nas parcelas a partir da diferença entre `data_prevista_pagamento` e `data_real_pagamento`.

* Para cada cliente são analisados os contratos históricos remanescentes, excluindo o contrato representativo utilizado para compor a população ativa.

* Com base no maior atraso observado ao longo do histórico, são construídas as seguintes flags de comportamento:

  * `ever_30`: cliente apresentou pelo menos um atraso superior a 30 dias;
  * `ever_45`: cliente apresentou pelo menos um atraso superior a 45 dias;
  * `ever_60`: cliente apresentou pelo menos um atraso superior a 60 dias;
  * `ever_90`: cliente apresentou pelo menos um atraso superior a 90 dias.

* Essas definições representam diferentes níveis de severidade da inadimplência e são avaliadas comparativamente para identificar o melhor equilíbrio entre relevância de negócio e volume de eventos disponíveis para modelagem.

Ao final da análise, uma das definições é selecionada como target oficial do projeto e utilizada nas etapas posteriores de desenvolvimento do modelo.


### 3.1 Possíveis definições de target

Vamos comparar as flags `ever_30`, `ever_45`, `ever_60` e `ever_90` para avaliar diferentes níveis de severidade de inadimplência e selecionar a definição mais adequada para modelagem.

In [12]:
population_target = build_population_target(population_active)

base_target = (
    population_active[
        ["id_cliente", "safra_mes"]
    ]
    .merge(
        population_target[
            [
                "id_cliente",
                "safra_mes",
                "ever_30",
                "ever_45",
                "ever_60",
                "ever_90",
            ]
        ],
        on=["id_cliente", "safra_mes"],
        how="left"
    )
)

summary = []

for col in ["ever_30", "ever_45", "ever_60", "ever_90"]:

    total = len(base_target)

    bad = base_target[col].fillna(False).sum()
    good = total - bad

    summary.append({
        "definition": col,
        "total": total,
        "good": good,
        "bad": bad,
        "% bad": round((bad / total) * 100, 2)
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

,definition,total,good,bad,% bad
0,ever_30,81882,80783,1099,1.34
1,ever_45,81882,81362,520,0.64
2,ever_60,81882,81440,442,0.54
3,ever_90,81882,81581,301,0.37


**Escolha do Target**

A definição do target deve equilibrar dois objetivos:

1. Representar um evento de inadimplência suficientemente severo para refletir risco de crédito real;
2. Manter volume adequado de eventos para permitir o treinamento de modelos estatisticamente robustos.

A análise das definições candidatas mostrou comportamentos distintos:

- **ever_30** apresenta maior volume de eventos, porém tende a capturar atrasos operacionais, negociações temporárias e comportamentos que nem sempre representam deterioração efetiva do risco de crédito.
- **ever_90** representa um evento extremamente severo, mas possui baixa incidência na população, reduzindo significativamente a quantidade de exemplos positivos disponíveis para treinamento.
- **ever_60** já apresenta uma redução importante na volumetria de eventos, tornando a classe positiva relativamente escassa e aumentando o risco de instabilidade estatística durante a modelagem.
- **ever_45** surge como o melhor equilíbrio entre severidade e representatividade. A definição mantém um volume de eventos suficiente para aprendizado do modelo, ao mesmo tempo em que evita classificar como inadimplentes clientes que apresentaram apenas atrasos curtos ou pontuais.

Dessa forma, a definição **`ever_45`** foi adotada como target final do projeto, sendo considerada a melhor combinação entre relevância de negócio e robustez estatística.

### 3.2 Target final com filtro na população ativa

Construímos o target final em nível de cliente utilizando a população ativa e os históricos de contratos e parcelas.


In [13]:
population_target = build_population_target(population_active)

area_target = population_target[
    [
        "id_cliente",
        "safra_mes",
        "target",
        "max_delay",
        "ever_30",
        "ever_45",
        "ever_60",
        "ever_90",
    ]
].copy()

population_active = population_active.merge(
    area_target,
    on=["id_cliente", "safra_mes"],
    how="left"
)

population_active["target"] = (
    population_active["target"]
    .fillna(0)
    .astype(int)
)

population_active = (
    population_active
    .sort_values(["safra_mes", "id_cliente"])
    .reset_index(drop=True)
)

display(population_active.head())

print("Target shape:", population_active.shape)


,id_cliente,data_solicitacao,dia_semana_solicitacao_submissao,hora_solicitacao_submissao,tipo_contrato_submissao,valor_credito_submissao,valor_bem_submissao,valor_parcela_submissao,sexo,data_nascimento,...,qtd_contratos_aceitos_historico,qtd_contratos_recusados_historico,soma_valor_credito_ativo_historico,media_valor_credito_aceito_historico,target,max_delay,ever_30,ever_45,ever_60,ever_90
0,100949,2020-01-18,SATURDAY,12,Consumer loans,96525.0,96525.0,10762.560,F,1987-03-09,...,2,0,178564.5,89282.25,0,-3.0,False,False,False,False
1,101032,2020-01-29,WEDNESDAY,13,Cash loans,92970.0,90000.0,18341.100,F,1966-11-09,...,4,1,462366.0,115591.50,0,-2.0,False,False,False,False
2,101294,2020-01-31,FRIDAY,15,Cash loans,164223.0,135000.0,18571.995,F,1980-07-14,...,2,0,236650.5,118325.25,0,-1.0,False,False,False,False
3,101684,2020-01-26,SUNDAY,14,Consumer loans,122773.5,115173.0,14851.035,F,1997-09-01,...,1,0,122773.5,122773.50,0,18.0,False,False,False,False
4,101813,2020-01-30,THURSDAY,12,Consumer loans,58702.5,60255.0,8007.030,F,1966-12-22,...,3,0,330255.0,110085.00,0,6.0,False,False,False,False


Target shape: (81882, 34)


## 4. Divisão de Treino e OOT

Dividimos o dataset em treino e OOT com base na safra, e em seguida extraímos um conjunto de teste e OOS a partir do treino.


In [14]:
train_base = population_active[population_active["safra_mes"].isin(TRAIN_PERIODS)].copy()
oot_base = population_active[population_active["safra_mes"].isin(OOT_PERIODS)].copy()

train_base = train_base.reset_index(drop=True)
oot_base = oot_base.reset_index(drop=True)

### 4.1 Volumetria total

In [15]:

print("Treino:", train_base.shape)
print("OOT:", oot_base.shape)

Treino: (61912, 34)
OOT: (19970, 34)


### 4.2 Distribuição de target por base

Mostramos a distribuição total de `target=1` (bad) e a distribuição por safra em cada base.


In [16]:
from src.target import print_target_distribution

for name, df in [
    ("Treino Base Geral", train_base),
    ("OOT", oot_base),
]:
    print_target_distribution(df, name)



--- Distribuição Target: Treino Base Geral ---
  safra  total  good  bad  % bad
2020-01    630   606   24   3.81
2020-02    672   661   11   1.64
2020-03    707   688   19   2.69
2020-04    729   707   22   3.02
2020-05    739   728   11   1.49
2020-06    737   728    9   1.22
2020-07    672   664    8   1.19
2020-08    668   661    7   1.05
2020-09    773   764    9   1.16
2020-10    864   858    6   0.69
2020-11    838   828   10   1.19
2020-12    818   808   10   1.22
2021-01    932   924    8   0.86
2021-02    806   802    4   0.50
2021-03    884   878    6   0.68
2021-04    813   809    4   0.49
2021-05    867   862    5   0.58
2021-06    854   842   12   1.41
2021-07    879   866   13   1.48
2021-08    924   911   13   1.41
2021-09    936   927    9   0.96
2021-10    946   936   10   1.06
2021-11    962   947   15   1.56
2021-12   1029  1017   12   1.17
2022-01   1165  1156    9   0.77
2022-02   1188  1174   14   1.18
2022-03   1310  1287   23   1.76
2022-04   1265  1253   12   

## 5. Persistência

Após a definição do target e a segmentação temporal da base, os datasets finais utilizados nas etapas de modelagem são persistidos na camada `processed`.

Essa abordagem garante reprodutibilidade, reduz o tempo de processamento das próximas etapas e assegura que todos os notebooks subsequentes utilizem exatamente a mesma definição de população e target.

Os seguintes artefatos são armazenados:

- `train_base.parquet`: base de desenvolvimento contendo as observações utilizadas para treinamento, validação e testes internos do modelo.
- `oot_base.parquet`: base Out-of-Time (OOT), reservada para avaliação temporal independente da performance do modelo.

Essas bases passam a representar a fonte oficial para as etapas de EDA, Feature Engineering e Modelagem.

In [17]:
train_base.to_parquet(
    PROCESSED_PATH / "train_base.parquet",
    index=False
)

oot_base.to_parquet(
    PROCESSED_PATH / "oot_base.parquet",
    index=False
)

print("Bases train_base e oot_base persistidas com sucesso.")

Bases train_base e oot_base persistidas com sucesso.


## 6. Conclusões

Nesta etapa foi realizada a construção do target de inadimplência e a preparação das bases que serão utilizadas durante o desenvolvimento do modelo.

Foram avaliadas diferentes definições de atraso (`ever_30`, `ever_45`, `ever_60` e `ever_90`) considerando simultaneamente severidade do evento e volume de observações disponíveis para modelagem.

A definição `ever_45` foi selecionada como target final por apresentar o melhor equilíbrio entre relevância de negócio e representatividade estatística, permitindo a construção de um modelo mais robusto e estável.

Após a definição do target, a população foi segmentada em bases de desenvolvimento e validação temporal (OOT), que foram persistidas para utilização nas próximas etapas do projeto.

No próximo notebook será realizada a Análise Exploratória dos Dados (EDA), com foco na compreensão das características da população, qualidade das variáveis e comportamento do evento de inadimplência.